<a href="https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

In Week 5 I built a model that ranks pages by how likely they are to lose clicks next week. This notebook audits my own work the way we audited the FlyRank research paper: where do the labels come from, does the validation design support the claim, and which of my own sentences go further than the evidence?

## 1. Two paper findings + my methodology questions

I picked two findings from *The State of AI-Driven SEO in Numbers* (March 2026). The paper is clear about its limits: it calls the work observational and the ML pages exploratory. My questions are about what it would take to make the claims stronger, the way I would want a reviewer to read my own work.

**Finding A: "Refreshing a mature page lifts health 3.2x and impressions 57x" (Finding 4, the freshness analysis).**
The paper reports that content older than 365 days, refreshed in the last 30 days, goes from 10.7 to 34.5 on health score and from 71 to 4,039 impressions.

- *Where does the "refreshed" group come from?* If pages were chosen for a refresh because they already looked promising (for example, they had some impressions), then part of the lift comes from the choice and not from the refresh itself. How were refresh candidates picked?
- *How many pages are in each group?* The 57x jump starts from 71 impressions, which is small. The group sizes would help a reader judge how stable the number is. The paper already flags a similar small-sample issue for the 361+ day bucket (283:1 from one declining page), which I appreciate.
- *Does a before/after on the same pages exist?* A comparison of the same pages before and after the refresh would be stronger than comparing different groups.

**Finding B: "Logistic regression separates growing from declining pages with 71% holdout accuracy" (ML appendix).**
The paper presents this as descriptive and exploratory, with an 80/20 split.

- *What is the base rate?* In Finding 1, about 62% of trending pages are rising. If the ML sample looks similar, a model that always says "rising" would already score about 62%, so 71% would be about 9 points of skill. The paper does not show the naive baseline next to the number.
- *Is the split grouped by brand?* The data covers 57 brands, and pages from one brand share traits. If pages from the same brand were on both sides of the 80/20 split, the holdout score can look better than it would on a new brand.
- *Where does the label come from?* "Growing" and "declining" come from an impressions trend over a window. It would help to know whether any feature (such as impressions or age) overlaps that same window.

Below is the small calculation behind my question for Finding B.

In [5]:
# Base rate implied by the paper's Finding 1 (counts of rising vs falling pages)
rising, falling = 74_187, 45_272
paper_base = rising / (rising + falling)
claimed_accuracy = 0.71

print(f"Share of pages rising in Finding 1: {paper_base:.1%}")
print(f"Accuracy of always guessing 'rising': about {paper_base:.1%}")
print(f"Claimed holdout accuracy: {claimed_accuracy:.0%} -> about {(claimed_accuracy - paper_base) * 100:.0f} points above the naive guess")
print("Assumption: the ML sample has a similar rising share. The paper does not report it for that sample.")

Share of pages rising in Finding 1: 62.1%
Accuracy of always guessing 'rising': about 62.1%
Claimed holdout accuracy: 71% -> about 9 points above the naive guess
Assumption: the ML sample has a similar rising share. The paper does not report it for that sample.


## 2. My model under an honest split (before/after)

My Week-5 model asks, for one page at one decision date: will its clicks over the next 7 days be at least 30% lower than over the previous 7 days? (Pages need at least 10 clicks in the previous week and complete 7-day windows on both sides.)

I compare three ways to test it. All use the same rows and the same metrics, and I print the base rate next to every score.

- **Before: random row split.** Shuffle all rows, train on 80%, test on 20%. This is what the default `train_test_split` would give.
- **After: grouped by client.** Five folds where every client appears on only one side. This is the question I care about: does it work on a client it has never seen?
- **Also: time-forward.** Train on the earliest decision dates and test on the latest. I leave one decision date out between them, because its label window would overlap the test features.

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from pathlib import Path

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

print("pandas", pd.__version__, "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

pandas 2.2.3 | numpy 2.4.6 | scikit-learn 1.6.1


In [3]:
# Warehouse connection. The token comes from Colab Secrets and is never printed.
%pip -q install duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    )
""")
print("Connected to the warehouse.")

Connected to the warehouse.


In [4]:
# Settings for the weekly frame. Change them here, nowhere else.
DECISION_DATES = ["2026-03-08", "2026-03-15", "2026-03-22", "2026-03-29"]  # weekly, windows do not overlap
MIN_PREV_CLICKS = 10      # page must have at least this many clicks in the previous 7 days
DROP_FRACTION = 0.30      # label = 1 if next-7-day clicks <= (1 - 0.30) * previous-7-day clicks

ROOT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
MONTHS = ["2026-02", "2026-03", "2026-04"]   # Feb for the "week before last", Apr for the outcome of late March
FILES = [f"{ROOT}/month={m}/*.parquet" for m in MONTHS]

first = pd.Timestamp(min(DECISION_DATES)) - pd.Timedelta(days=14)
last = pd.Timestamp(max(DECISION_DATES)) + pd.Timedelta(days=7)
date_list = ", ".join(f"DATE '{d}'" for d in DECISION_DATES)

sql = f"""
WITH d AS (
    SELECT UNNEST([{date_list}]) AS decision_date
),
base AS (
    SELECT
        report_date, client_hash_id, content_hash_id,
        gsc_clicks, gsc_impressions,
        NULLIF(gsc_avg_position, 0) AS pos,
        CASE WHEN ga4_data_available IS TRUE THEN sessions_organic END AS organic,
        CASE WHEN ga4_data_available IS TRUE THEN scroll_events END AS scroll,
        (ga4_data_available IS TRUE) AS ga4_ok
    FROM read_parquet({FILES})
    WHERE gsc_data_available IS TRUE          -- zero-filled rows are not real zeros
      AND report_date BETWEEN DATE '{first.date()}' AND DATE '{last.date()}'
),
agg AS (
    SELECT
        d.decision_date, b.client_hash_id, b.content_hash_id,

        COUNT(*)              FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_days,
        SUM(b.gsc_clicks)     FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_clicks,
        SUM(b.gsc_impressions) FILTER (WHERE b.report_date BETWEEN d.decision_date - 7 AND d.decision_date - 1) AS prev_impr,
        AVG(b.pos)            FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_pos,
        COUNT(*)              FILTER (WHERE b.ga4_ok AND b.report_date BETWEEN d.decision_date - 7 AND d.decision_date - 1) AS prev_ga4_days,
        SUM(b.organic)        FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_organic,
        SUM(b.scroll)         FILTER (WHERE b.report_date BETWEEN d.decision_date - 7  AND d.decision_date - 1) AS prev_scroll,

        COUNT(*)              FILTER (WHERE b.report_date BETWEEN d.decision_date - 14 AND d.decision_date - 8) AS prior_days,
        SUM(b.gsc_clicks)     FILTER (WHERE b.report_date BETWEEN d.decision_date - 14 AND d.decision_date - 8) AS prior_clicks,

        COUNT(*)              FILTER (WHERE b.report_date BETWEEN d.decision_date + 1  AND d.decision_date + 7) AS future_days,
        SUM(b.gsc_clicks)     FILTER (WHERE b.report_date BETWEEN d.decision_date + 1  AND d.decision_date + 7) AS future_clicks
    FROM d
    JOIN base b ON b.report_date BETWEEN d.decision_date - 14 AND d.decision_date + 7
    GROUP BY d.decision_date, b.client_hash_id, b.content_hash_id
)
SELECT * FROM agg
WHERE prev_days = 7 AND future_days = 7       -- complete windows only
  AND prev_clicks >= {MIN_PREV_CLICKS}
"""

CACHE = Path("w05_weekly_frame.parquet")   # local cache only, never committed (parquet is gitignored)
if CACHE.exists():
    raw = pd.read_parquet(CACHE)
    print("Loaded cached frame:", CACHE)
else:
    raw = con.sql(sql).df()
    raw.to_parquet(CACHE, index=False)
    print("Queried the warehouse and cached the result.")

print("Rows:", f"{len(raw):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queried the warehouse and cached the result.
Rows: 16,446


In [6]:
frame = raw.copy()
frame["decision_date"] = pd.to_datetime(frame["decision_date"])
for c in ["prev_clicks", "prev_impr", "prior_clicks", "future_clicks", "prev_organic", "prev_scroll"]:
    frame[c] = frame[c].astype(float)

# Features that exist at the decision moment
frame["prev_ctr"] = frame["prev_clicks"] / frame["prev_impr"].replace(0, np.nan)
has_prior = frame["prior_days"] == 7
frame["has_prior"] = has_prior.astype(int)
frame["prior_clicks"] = frame["prior_clicks"].where(has_prior)
frame["clicks_change"] = ((frame["prev_clicks"] - frame["prior_clicks"]) / (frame["prior_clicks"] + 1)).clip(-1, 5)
frame["has_position"] = frame["prev_pos"].notna().astype(int)

# Label: a drop of at least DROP_FRACTION (uses future clicks, so it is never a feature)
frame["y"] = (frame["future_clicks"] <= (1 - DROP_FRACTION) * frame["prev_clicks"]).astype(int)

frame = frame.sort_values(["decision_date", "client_hash_id", "content_hash_id"]).reset_index(drop=True)

FEATURES = ["prev_clicks", "prev_impr", "prev_ctr", "prev_pos", "prior_clicks", "clicks_change",
            "prev_organic", "prev_scroll", "prev_ga4_days", "has_prior", "has_position"]

print("Eligible page-weeks:", f"{len(frame):,}")
print("Clients:", frame["client_hash_id"].nunique(), "| distinct pages:", frame[["client_hash_id", "content_hash_id"]].drop_duplicates().shape[0])
print("Base rate (share of pages that dropped):", round(frame["y"].mean(), 3))
print()
print(frame.groupby("decision_date")["y"].agg(rows="size", base_rate="mean").round(3))
print()
print("Missing values in features:")
print(frame[FEATURES].isna().mean().round(3).to_string())

Eligible page-weeks: 16,446
Clients: 26 | distinct pages: 7627
Base rate (share of pages that dropped): 0.385

               rows  base_rate
decision_date                 
2026-03-08     3729      0.417
2026-03-15     3787      0.422
2026-03-22     3979      0.340
2026-03-29     4951      0.370

Missing values in features:
prev_clicks      0.000
prev_impr        0.000
prev_ctr         0.000
prev_pos         0.000
prior_clicks     0.082
clicks_change    0.082
prev_organic     0.478
prev_scroll      0.478
prev_ga4_days    0.000
has_prior        0.000
has_position     0.000


In [7]:
from sklearn.model_selection import GroupKFold, ShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

K_SMALL, K_LARGE = 50, 500
TRAIN_CAP = 150_000


def precision_at_k(y, score, k):
    k = min(k, len(y))
    top = np.argsort(-score, kind="stable")[:k]
    return float(np.mean(y[top]))


def evaluate(y, score):
    return {f"P@{K_SMALL}": precision_at_k(y, score, K_SMALL),
            f"P@{K_LARGE}": precision_at_k(y, score, K_LARGE),
            "AP": average_precision_score(y, score),
            "ROC-AUC": roc_auc_score(y, score)}


def baseline_score(train_df, test_df):
    # Week 4 rule (70% CTR gap + 30% visibility) on the 7-day window; CTR reference from training rows only
    ctr_ref = train_df.loc[train_df["prev_ctr"] > 0, "prev_ctr"].median()
    vis_max = np.log1p(train_df["prev_impr"].max())
    ctr_opportunity = (1 - test_df["prev_ctr"].fillna(0) / ctr_ref).clip(0, 1)
    visibility = np.log1p(test_df["prev_impr"]) / vis_max
    return (0.7 * ctr_opportunity + 0.3 * visibility).to_numpy()


def linear_features(df):
    out = pd.DataFrame(index=df.index)
    for c in ["prev_clicks", "prev_impr", "prior_clicks", "prev_organic", "prev_scroll"]:
        out[f"log_{c}"] = np.log1p(df[c].fillna(0))
    for c in ["prev_ctr", "prev_pos", "clicks_change", "prev_ga4_days", "has_prior", "has_position"]:
        out[c] = df[c]
    return out


def raw_features(df):
    return df[FEATURES]


MODELS = {
    "Logistic Regression": (
        lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                              LogisticRegression(max_iter=1000, random_state=SEED)),
        linear_features),
    "Gradient Boosting": (
        lambda: HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200,
                                               min_samples_leaf=200, l2_regularization=1.0,
                                               random_state=SEED),
        raw_features),
}


def run_split(train_idx, test_idx):
    tr_df, te_df = frame.iloc[train_idx], frame.iloc[test_idx]
    y_te = te_df["y"].to_numpy()
    scores = {"Week 4 baseline": baseline_score(tr_df, te_df)}
    tr_fit = tr_df if len(tr_df) <= TRAIN_CAP else tr_df.sample(TRAIN_CAP, random_state=SEED)
    for name, (make, prep) in MODELS.items():
        model = make().fit(prep(tr_fit), tr_fit["y"])
        scores[name] = model.predict_proba(prep(te_df))[:, 1]
    metrics = {name: evaluate(y_te, s) for name, s in scores.items()}
    return metrics, scores, float(y_te.mean())


results = []
oof_scores = {name: pd.Series(np.nan, index=frame.index) for name in ["Week 4 baseline", *MODELS]}

# BEFORE: random 80/20 row split, repeated 5 times
for rep, (tr, te) in enumerate(ShuffleSplit(n_splits=5, test_size=0.2, random_state=SEED).split(frame)):
    m, _, base = run_split(tr, te)
    for name, vals in m.items():
        results.append({"split": "Random rows (before)", "rep": rep, "model": name, "base_rate": base, **vals})

# AFTER: grouped by client, 5 folds (these out-of-fold scores are reused for the error analysis below)
group_folds = list(GroupKFold(n_splits=5).split(frame, groups=frame["client_hash_id"]))
for rep, (tr, te) in enumerate(group_folds):
    assert set(frame.iloc[tr]["client_hash_id"]).isdisjoint(set(frame.iloc[te]["client_hash_id"]))
    m, s, base = run_split(tr, te)
    for name, vals in m.items():
        results.append({"split": "Grouped by client (after)", "rep": rep, "model": name, "base_rate": base, **vals})
        oof_scores[name].iloc[te] = s[name]

# ALSO: time-forward with one purged decision date between train and test
dates = sorted(frame["decision_date"].unique())
train_dates, purge_date, test_date = dates[:2], dates[2], dates[3]
tr = np.where(frame["decision_date"].isin(train_dates))[0]
te = np.where(frame["decision_date"] == test_date)[0]
m, _, base = run_split(tr, te)
for name, vals in m.items():
    results.append({"split": "Time-forward", "rep": 0, "model": name, "base_rate": base, **vals})
print("Time-forward: train on", [str(pd.Timestamp(d).date()) for d in train_dates],
      "| skipped", str(pd.Timestamp(purge_date).date()), "| test on", str(pd.Timestamp(test_date).date()))

results = pd.DataFrame(results)

Time-forward: train on ['2026-03-08', '2026-03-15'] | skipped 2026-03-22 | test on 2026-03-29


In [8]:
metric_cols = [f"P@{K_SMALL}", f"P@{K_LARGE}", "AP", "ROC-AUC"]
order = ["Week 4 baseline", *MODELS]
split_order = ["Random rows (before)", "Grouped by client (after)", "Time-forward"]

table = (results.groupby(["split", "model"])[["base_rate", *metric_cols]].mean()
         .reindex(pd.MultiIndex.from_product([split_order, order], names=["split", "model"])))
print("Mean over repeats/folds. base_rate = share of pages that dropped in the test rows (what a random list scores).")
table.round(3)

Mean over repeats/folds. base_rate = share of pages that dropped in the test rows (what a random list scores).


base_rate   P@50  P@500     AP  \
split                     model                                                 
Random rows (before)      Week 4 baseline          0.383  0.312  0.381  0.366   
                          Logistic Regression      0.383  0.676  0.575  0.500   
                          Gradient Boosting        0.383  0.720  0.610  0.556   
Grouped by client (after) Week 4 baseline          0.389  0.344  0.372  0.367   
                          Logistic Regression      0.389  0.664  0.549  0.502   
                          Gradient Boosting        0.389  0.696  0.505  0.495   
Time-forward              Week 4 baseline          0.370  0.380  0.380  0.364   
                          Logistic Regression      0.370  0.760  0.676  0.442   
                          Gradient Boosting        0.370  0.800  0.314  0.394   

                                               ROC-AUC  
split                     model                         
Random rows (before)      Week 4 baseline        0.454  
                          Logistic Regression    0.626  
                          Gradient Boosting      0.690  
Grouped by client (after) Week 4 baseline        0.440  
                          Logistic Regression    0.611  
                          Gradient Boosting      0.602  
Time-forward              Week 4 baseline        0.445  
                          Logistic Regression    0.541  
                          Gradient Boosting      0.510

In [9]:
# The gap between the random split and the honest split is itself a finding
mean_by = results.groupby(["split", "model"])[metric_cols].mean()
gap = (mean_by.loc["Random rows (before)"] - mean_by.loc["Grouped by client (after)"]).loc[list(MODELS)]
print("Random-split score minus grouped-split score (positive = the random split looked better than it should):")
display(gap.round(3))

# Why: how much can the model memorise under a random split?
tr_idx, te_idx = next(ShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(frame))
pair = frame["client_hash_id"] + "|" + frame["content_hash_id"]
seen_pair = pair.iloc[te_idx].isin(set(pair.iloc[tr_idx])).mean()
seen_client = frame["client_hash_id"].iloc[te_idx].isin(set(frame["client_hash_id"].iloc[tr_idx])).mean()
print(f"Random split: {seen_pair:.1%} of test rows are a page the model already saw in training (another week), "
      f"{seen_client:.1%} are from a client it already saw.")
print("Grouped split: 0.0% of test rows are from a client seen in training (checked with an assert above).")

Random-split score minus grouped-split score (positive = the random split looked better than it should):


,P@50,P@500,AP,ROC-AUC
model,,,,
Logistic Regression,0.012,0.026,-0.001,0.015
Gradient Boosting,0.024,0.106,0.061,0.089


Random split: 73.0% of test rows are a page the model already saw in training (another week), 100.0% are from a client it already saw.
Grouped split: 0.0% of test rows are from a client seen in training (checked with an assert above).


### Where the honest model fails

Numbers hide how a model goes wrong, so here are the real rows behind its mistakes. These use the out-of-fold scores from the grouped-by-client run, so every page was scored by a model that never saw its client. Clients are relabelled C1, C2, ... and no page ids are shown.

In [10]:
learned = list(MODELS)
best_name = results[results["split"] == "Grouped by client (after)"].groupby("model")["AP"].mean().loc[learned].idxmax()
print("Best learned model on held-out clients (mean AP):", best_name)

view = frame.copy()
view["score"] = oof_scores[best_name]
view["fold"] = -1
for i, (_, te) in enumerate(group_folds):
    view.iloc[te, view.columns.get_loc("fold")] = i

# top-500 picks in every fold
picked_idx = np.concatenate([view[view["fold"] == i].sort_values("score", ascending=False).head(K_LARGE).index for i in range(5)])
view["picked"] = view.index.isin(picked_idx)

view["size_band"] = pd.qcut(view["prev_clicks"], 4, labels=["smallest 25%", "25-50%", "50-75%", "largest 25%"])
all_pages = view.groupby("size_band", observed=True)
chosen = view[view["picked"]].groupby("size_band", observed=True)
by_size = pd.DataFrame({"share_of_all_pages": all_pages.size() / len(view),
                        "share_of_top_picks": chosen.size() / view["picked"].sum(),
                        "drop_rate_all_pages": all_pages["y"].mean(),
                        "drop_rate_in_picks": chosen["y"].mean()})
print("By page size (previous-week clicks)")
display(by_size.round(3))

rows = []
for cid, g in view.groupby("client_hash_id"):
    if len(g) >= 300 and 0 < g["y"].sum() < len(g):
        rows.append({"client": cid, "rows": len(g), "base_rate": g["y"].mean(), "AP": average_precision_score(g["y"], g["score"])})
by_client = pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)
by_client["client"] = [f"C{i+1}" for i in range(len(by_client))]
by_client["AP_minus_base_rate"] = by_client["AP"] - by_client["base_rate"]
print("Average precision per client (clients with at least 300 rows)")
display(by_client.round(3))

show = ["decision_date", "prev_clicks", "prior_clicks", "future_clicks", "prev_ctr", "prev_pos", "score"]
fa = view[view["picked"] & (view["y"] == 0)].sort_values("score", ascending=False).head(3)
print("Most confident false alarms (picked in the top 500, but clicks did not fall enough)")
display(fa[show].round(3))
ms = view[view["y"] == 1].copy()
ms["clicks_lost"] = ms["prev_clicks"] - ms["future_clicks"]
print("Biggest misses (large drops that the model scored low)")
display(ms.sort_values("clicks_lost", ascending=False).head(3)[show + ["clicks_lost"]].round(3))
share_clients_worse = (by_client["AP_minus_base_rate"] <= 0).mean() if len(by_client) else float("nan")
print(f"Clients where the model did no better than the base rate: {share_clients_worse:.0%} of {len(by_client)}")

Best learned model on held-out clients (mean AP): Logistic Regression
By page size (previous-week clicks)


,share_of_all_pages,share_of_top_picks,drop_rate_all_pages,drop_rate_in_picks
size_band,,,,
smallest 25%,0.260,0.375,0.385,0.519
25-50%,0.256,0.294,0.438,0.622
50-75%,0.248,0.196,0.411,0.574
largest 25%,0.236,0.134,0.301,0.435


Average precision per client (clients with at least 300 rows)


,client,rows,base_rate,AP,AP_minus_base_rate
0,C1,4912,0.363,0.544,0.181
1,C2,2876,0.453,0.548,0.096
2,C3,2648,0.420,0.607,0.188
3,C4,1510,0.391,0.502,0.110
4,C5,1226,0.117,0.124,0.007
5,C6,970,0.413,0.574,0.160
6,C7,865,0.439,0.554,0.114
7,C8,724,0.362,0.408,0.046


Most confident false alarms (picked in the top 500, but clicks did not fall enough)


,decision_date,prev_clicks,prior_clicks,future_clicks,prev_ctr,prev_pos,score
3773,2026-03-15,15.0,0.0,23.0,0.01,9.637,0.909
3777,2026-03-15,11.0,0.0,22.0,0.02,6.266,0.904
10966,2026-03-22,1277.0,NaN,1589.0,0.01,2.410,0.887


Biggest misses (large drops that the model scored low)


,decision_date,prev_clicks,prior_clicks,future_clicks,prev_ctr,prev_pos,score,clicks_lost
13225,2026-03-29,488.0,111.0,36.0,0.007,12.005,0.402,452.0
10848,2026-03-22,318.0,NaN,25.0,0.009,3.202,0.803,293.0
12378,2026-03-29,704.0,639.0,453.0,0.013,4.515,0.066,251.0


Clients where the model did no better than the base rate: 0% of 8


## 3. Leakage audit

The same hunt as Week 3, now on my final feature set. The three leak types from the skill: features that come from the label, features whose time window overlaps the label window, and features copied from an existing product score or flag.

**Timeline.** Every feature must be known at the decision moment, and the label must be strictly after it.

In [11]:
print("""
 D-14 ... D-8   prior week        -> prior_clicks, clicks_change       (features)
 D-7  ... D-1   previous week     -> prev_* columns                    (features)
 D              decision day      -> not used anywhere
 D+1  ... D+7   next week         -> future_clicks                     (label only)
""")

feature_sources = pd.DataFrame([
    ("prev_clicks",   "D-7 to D-1",  "yes", "Also used in the label rule (a drop is measured against it), so I test it below"),
    ("prev_impr",     "D-7 to D-1",  "yes", ""),
    ("prev_ctr",      "D-7 to D-1",  "yes", "clicks / impressions of the same week"),
    ("prev_pos",      "D-7 to D-1",  "yes", "mean of daily average position"),
    ("prior_clicks",  "D-14 to D-8", "yes", "only filled when all 7 days are present"),
    ("clicks_change", "D-14 to D-1", "yes", "previous week vs the week before"),
    ("prev_organic",  "D-7 to D-1",  "yes", "GA4 only, empty when GA4 is not available"),
    ("prev_scroll",   "D-7 to D-1",  "yes", "GA4 only"),
    ("prev_ga4_days", "D-7 to D-1",  "yes", "count of GA4 days"),
    ("has_prior",     "D-14 to D-8", "yes", "flag"),
    ("has_position",  "D-7 to D-1",  "yes", "flag"),
], columns=["feature", "window", "known at decision time?", "note"])
assert set(feature_sources["feature"]) == set(FEATURES)
feature_sources


 D-14 ... D-8   prior week        -> prior_clicks, clicks_change       (features)
 D-7  ... D-1   previous week     -> prev_* columns                    (features)
 D              decision day      -> not used anywhere
 D+1  ... D+7   next week         -> future_clicks                     (label only)



,feature,window,known at decision time?,note
0,prev_clicks,D-7 to D-1,yes,Also used in the label rule (a drop is measure...
1,prev_impr,D-7 to D-1,yes,
2,prev_ctr,D-7 to D-1,yes,clicks / impressions of the same week
3,prev_pos,D-7 to D-1,yes,mean of daily average position
4,prior_clicks,D-14 to D-8,yes,only filled when all 7 days are present
5,clicks_change,D-14 to D-1,yes,previous week vs the week before
6,prev_organic,D-7 to D-1,yes,"GA4 only, empty when GA4 is not available"
7,prev_scroll,D-7 to D-1,yes,GA4 only
8,prev_ga4_days,D-7 to D-1,yes,count of GA4 days
9,has_prior,D-14 to D-8,yes,flag


In [12]:
# Check 1: does any single feature separate the label suspiciously well? (no model, whole frame)
rows = []
for f in FEATURES:
    x = frame[f].astype(float)
    x = x.fillna(x.median())
    a = roc_auc_score(frame["y"], x)
    rows.append({"feature": f, "AUC (one feature alone)": a, "strength (distance from 0.5)": abs(a - 0.5)})
single = pd.DataFrame(rows).sort_values("strength (distance from 0.5)", ascending=False)
display(single.round(3))
max_single = single["AUC (one feature alone)"].sub(0.5).abs().max() + 0.5
print(f"Strongest single feature reaches AUC {max_single:.3f}. A value near 1.0 would be a leak; one around 0.6 is ordinary signal.")

,feature,AUC (one feature alone),strength (distance from 0.5)
4,prior_clicks,0.391,0.109
5,clicks_change,0.607,0.107
1,prev_impr,0.434,0.066
0,prev_clicks,0.453,0.047
6,prev_organic,0.460,0.040
2,prev_ctr,0.537,0.037
8,prev_ga4_days,0.532,0.032
7,prev_scroll,0.483,0.017
9,has_prior,0.483,0.017
3,prev_pos,0.513,0.013


Strongest single feature reaches AUC 0.609. A value near 1.0 would be a leak; one around 0.6 is ordinary signal.


In [13]:
# Check 2: plant a leak on purpose. If the score does not jump, my test harness is broken.
tr0, te0 = group_folds[0]
make_gb = MODELS["Gradient Boosting"][0]

def fold0_ap(X):
    tr_rows = tr0 if len(tr0) <= TRAIN_CAP else np.random.default_rng(SEED).choice(tr0, TRAIN_CAP, replace=False)
    m = make_gb().fit(X.iloc[tr_rows], frame["y"].iloc[tr_rows])
    return average_precision_score(frame["y"].iloc[te0], m.predict_proba(X.iloc[te0])[:, 1])

clean_ap = fold0_ap(frame[FEATURES])
leaky = frame[FEATURES].copy()
leaky["planted_leak"] = frame["future_clicks"] / frame["prev_clicks"]   # reads next week's clicks on purpose
leak_ap = fold0_ap(leaky)
print(f"Fold 0 AP without the planted leak: {clean_ap:.3f}")
print(f"Fold 0 AP with the planted leak:    {leak_ap:.3f}   <- harness works if this jumps toward 1.0")

Fold 0 AP without the planted leak: 0.543
Fold 0 AP with the planted leak:    1.000   <- harness works if this jumps toward 1.0


In [14]:
# Check 3: train with and without the features closest to the label rule
suspects = ["prev_clicks", "prior_clicks", "clicks_change"]
make_gb = MODELS["Gradient Boosting"][0]

def grouped_ap(cols):
    aps = []
    for tr, te in group_folds:
        tr_rows = tr if len(tr) <= TRAIN_CAP else np.random.default_rng(SEED).choice(tr, TRAIN_CAP, replace=False)
        m = make_gb().fit(frame[cols].iloc[tr_rows], frame["y"].iloc[tr_rows])
        aps.append(average_precision_score(frame["y"].iloc[te], m.predict_proba(frame[cols].iloc[te])[:, 1]))
    return float(np.mean(aps))

ap_all = grouped_ap(FEATURES)
ap_without = grouped_ap([f for f in FEATURES if f not in suspects])
print(f"Grouped AP with all features:           {ap_all:.3f}")
print(f"Grouped AP without {suspects}: {ap_without:.3f}")
base_now = frame["y"].mean()
carried = (ap_all - ap_without) / max(ap_all - base_now, 1e-9)
print(f"Base rate: {base_now:.3f}. These features carry about {carried:.0%} of the model's lift above the base rate.")
if ap_without <= base_now + 0.02:
    print("Without them the model falls to about the base rate. They are known at decision time, so this is not a time leak, "
          "but the result rests on past click levels and momentum, and it may partly reflect regression to the mean.")
else:
    print("The model keeps real skill without them, so it does not depend on these features alone.")

# Check 4: pipeline sanity. With labels shuffled in training AND test, AUC should sit near 0.5.
rng_s = np.random.default_rng(SEED)
aucs = []
for _ in range(5):
    y_all = rng_s.permutation(frame["y"].to_numpy())
    tr_rows = rng_s.choice(tr0, min(len(tr0), 50_000), replace=False)
    m = make_gb().fit(frame[FEATURES].iloc[tr_rows], y_all[tr_rows])
    aucs.append(roc_auc_score(y_all[te0], m.predict_proba(frame[FEATURES].iloc[te0])[:, 1]))
auc_shuf = float(np.mean(aucs))
print(f"Shuffled labels, 5 repeats: mean AUC {auc_shuf:.3f} (should be close to 0.5)")

Grouped AP with all features:           0.495
Grouped AP without ['prev_clicks', 'prior_clicks', 'clicks_change']: 0.475
Base rate: 0.385. These features carry about 19% of the model's lift above the base rate.
The model keeps real skill without them, so it does not depend on these features alone.
Shuffled labels, 5 repeats: mean AUC 0.496 (should be close to 0.5)


In [15]:
checklist = pd.DataFrame([
    ("Timeline drawn, all features before the label window", "yes (windows set in the SQL, shown above)"),
    ("No label-derived columns in the features", "future_clicks is not a feature; prev_clicks is a feature AND part of the label rule, tested above"),
    ("No product flags or existing scores as features", "yes: the Week 4 rule is only a baseline, never an input"),
    ("Split grouped by client", "yes: asserted that no client is on both sides"),
    ("Base rate printed next to every metric", f"yes (random-list score is {frame['y'].mean():.3f})"),
    ("Planted leak makes the score jump", "passed" if leak_ap > clean_ap + 0.15 else "CHECK: score did not jump much"),
    ("No single feature looks too good", "passed" if max_single < 0.8 else "CHECK: one feature is very strong"),
    ("Shuffled labels give about 0.5", "passed" if 0.45 <= auc_shuf <= 0.55 else "CHECK: not near 0.5"),
    ("Metrics are out-of-fold", "yes"),
], columns=["check", "result"])
checklist

,check,result
0,"Timeline drawn, all features before the label ...","yes (windows set in the SQL, shown above)"
1,No label-derived columns in the features,future_clicks is not a feature; prev_clicks is...
2,No product flags or existing scores as features,"yes: the Week 4 rule is only a baseline, never..."
3,Split grouped by client,yes: asserted that no client is on both sides
4,Base rate printed next to every metric,yes (random-list score is 0.385)
5,Planted leak makes the score jump,passed
6,No single feature looks too good,passed
7,Shuffled labels give about 0.5,passed
8,Metrics are out-of-fold,yes


## 4. Claim rewrite

My boldest first draft of a claim was: *"My model predicts which pages will lose traffic, and it beats the Week 4 baseline."*

That sentence goes further than the evidence in at least four ways: it says "predicts" without a base rate, it does not say which split was used, it ignores how the random split flatters the score, and it says nothing about what the model cannot show. The cell below rewrites it using the numbers from this run, so the text cannot drift from the table.

In [16]:
g = results[results["split"] == "Grouped by client (after)"].groupby("model")[metric_cols + ["base_rate"]]
gm, gs = g.mean(), g.std()
rm = results[results["split"] == "Random rows (before)"].groupby("model")[metric_cols].mean()
tm = results[results["split"] == "Time-forward"].set_index("model")

base = frame["y"].mean()
b_p500 = gm.loc["Week 4 baseline", f"P@{K_LARGE}"]
m_p500 = gm.loc[best_name, f"P@{K_LARGE}"]
m_ap, r_ap = gm.loc[best_name, "AP"], rm.loc[best_name, "AP"]
sd_p500 = gs.loc[best_name, f"P@{K_LARGE}"]

if m_p500 - b_p500 > sd_p500:
    verdict = "ranks held-out clients' pages better than the Week 4 rule in this sample"
else:
    verdict = "does not clearly separate from the Week 4 rule in this sample (the gap is within fold-to-fold variation)"

print("DRAFT (too strong):")
print('  "My model predicts which pages will lose traffic, and it beats the Week 4 baseline."\n')
print("REWRITE (what the evidence supports):")
print(f"  On clients it had never seen, the {best_name} model's top {K_LARGE} pages dropped {m_p500:.1%} of the time, "
      f"compared with {base:.1%} for a random list and {b_p500:.1%} for the Week 4 rule (measured across 5 grouped folds).")
print(f"  Directionally, the model {verdict}.")
if r_ap > m_ap + 0.005:
    print(f"  A random row split would have reported AP {r_ap:.3f} instead of {m_ap:.3f} on held-out clients, "
          f"so the random split flattered the model. I treat the grouped number as the honest one.")
else:
    print(f"  In this sample the random split (AP {r_ap:.3f}) did not look better than the grouped split (AP {m_ap:.3f}), "
          f"so memorising clients and pages did not noticeably inflate the score. I still report the grouped number, because it matches how the model would be used on new clients.")
print(f"  On a forward-in-time test (last decision date), its top {K_LARGE} dropped {tm.loc[best_name, f'P@{K_LARGE}']:.1%} of the time "
      f"(base rate {tm.loc[best_name, 'base_rate']:.1%}).")
print("  This is decision-support for choosing which pages to review first. It is observed on four weekly dates in March 2026,"
      " and it does not show why a page dropped or that a refresh would bring clicks back.")

DRAFT (too strong):
  "My model predicts which pages will lose traffic, and it beats the Week 4 baseline."

REWRITE (what the evidence supports):
  On clients it had never seen, the Logistic Regression model's top 500 pages dropped 54.9% of the time, compared with 38.5% for a random list and 37.2% for the Week 4 rule (measured across 5 grouped folds).
  Directionally, the model ranks held-out clients' pages better than the Week 4 rule in this sample.
  In this sample the random split (AP 0.500) did not look better than the grouped split (AP 0.502), so memorising clients and pages did not noticeably inflate the score. I still report the grouped number, because it matches how the model would be used on new clients.
  On a forward-in-time test (last decision date), its top 500 dropped 67.6% of the time (base rate 37.0%).
  This is decision-support for choosing which pages to review first. It is observed on four weekly dates in March 2026, and it does not show why a page dropped or that a 

**Words I now avoid, and what I use instead**

| Instead of | I write |
|---|---|
| "predicts" / "knows" | "ranks", "observed", "measured" |
| "proves" / "shows that X causes Y" | "is associated with", "directionally suggests" |
| "accurate" (with no base rate) | the score next to the random-list score |
| "works" | "worked on held-out clients in this sample" |
| "drivers" (for feature importance) | "features the model relied on", with no claim about cause |

**Claims I am not making:** that the model works on June or any other month, that a refresh recovers lost clicks, that the label tells me why a page dropped, or that results from four weekly dates in one month generalise.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.